# Notebook 01 — Data Preparation

This notebook constructs the cleaned analysis dataset for the airline fare regression project.

The raw June 2026 DB1C Ticket file contains more than 13 million observations and is too large to load directly into memory. The notebook therefore uses chunk-based processing to inspect the raw data and draw a reproducible random sample from the full file.

The main tasks are to:

- inspect the structure and coding of the DB1C Ticket data;
- identify redundant, constant, and unsuitable variables;
- investigate unusual fare records;
- define the main commercial-fare analysis population;
- screen the retained variables for later modelling;
- construct and export the final regression-ready dataset.

The resulting prepared dataset is used as the common input for all subsequent notebooks.


## 1. Import & Configuration

This section imports the required Python libraries and defines the project directory structure.

The analysis uses the DB1C Ticket file as the primary data source. Because the raw dataset is several gigabytes in size, subsequent sections will process the file in chunks rather than loading the entire dataset into memory.

### 1.1 Import Libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

### 1.2 Configure Project Paths

In [ ]:
PROJECT_DIR = Path.cwd().resolve()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent


# Define project folders

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
INTERIM_DATA_DIR = PROJECT_DIR / "data" / "interim"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"


# Define the DB1C Ticket file

TICKET_FILE = RAW_DATA_DIR / "DB1C.TICKET.202606.16SEP2026.csv"

### 1.3 Verify File and Directory Configuration

Before processing the raw data, the configured project paths and Ticket file are checked to ensure that the required directories and source file are available.

In [ ]:
print(f"Project directory:   {PROJECT_DIR}")
print(f"Raw data directory:  {RAW_DATA_DIR}")
print(f"Interim directory:   {INTERIM_DATA_DIR}")
print(f"Processed directory: {PROCESSED_DATA_DIR}")

print("\nFile check:")
print(f"Ticket file exists:  {TICKET_FILE.exists()}")

if TICKET_FILE.exists():
    file_size_gb = TICKET_FILE.stat().st_size / (1024 ** 3)
    print(f"Ticket file size:    {file_size_gb:.2f} GB")

## 2. Initial Data Inspection

Before selecting variables or constructing the analysis sample, the raw DB1C Ticket file is inspected to understand its structure.

Only a small number of rows are loaded at this stage. This allows the column names, data types, and example observations to be examined without loading the full 2.52 GB dataset into memory.

### 2.1 Inspect Columns and Sample Records

In [ ]:
ticket_preview = pd.read_csv(
    TICKET_FILE,
    nrows=10,
    low_memory=False
)

print(f"Number of columns: {ticket_preview.shape[1]}")
print("\nColumn names:")
print(ticket_preview.columns.tolist())

display(ticket_preview)

#### Initial Inspection

The DB1C Ticket file contains 39 variables at the itinerary level. The available fields provide a broad set of potential predictors covering:

- itinerary structure and timing;
- origin and geographical information;
- reporting and issuing carriers;
- journey characteristics such as round-trip and online status;
- purchase-window information;
- ticket amount and related fare fields;
- travel-distance measures.

The preview also shows several very low `TotalAmount` values, including 5.60 and 17.00. These observations will be investigated systematically during data-quality assessment rather than removed based on the small preview alone.

Several variables appear to represent related concepts, particularly the different distance and carrier fields. Their suitability and potential redundancy will be assessed before constructing the modelling dataset.

### 2.2 Inspect Data Types

In [ ]:
ticket_dtypes = (
    ticket_preview.dtypes
    .rename("dtype")
    .to_frame()
)

display(ticket_dtypes)

In [ ]:
print("Data type counts:")
print(ticket_preview.dtypes.value_counts())

#### Interpretation

The Ticket file contains a mixture of numerical and categorical variables:

- 18 integer variables;
- 11 floating-point variables;
- 10 string variables.

The data types are broadly consistent with the expected structure of the dataset. Numerical variables include fare, distance, itinerary counts, and coded indicators, while categorical variables include airport, carrier, state, and purchase-window information.

Several variables that appear categorical are stored numerically, such as `RoundTrip`, `OnLine`, `ItinGeoType`, and `DistanceGroup`. Their actual values and coding should therefore be inspected before modelling.

The data types shown here are based on only 10 rows, so a larger sample is required to assess missing values, category frequencies, and whether any columns contain mixed or irregular values.

### 2.3 Inspect a Larger Data Chunk

In [ ]:
ticket_check = next(
    pd.read_csv(
        TICKET_FILE,
        chunksize=100_000,
        low_memory=False
    )
)

print(f"Rows loaded:    {ticket_check.shape[0]:,}")
print(f"Columns loaded: {ticket_check.shape[1]}")

In [ ]:
variable_summary = pd.DataFrame({
    "dtype": ticket_check.dtypes.astype(str),
    "missing_n": ticket_check.isna().sum(),
    "missing_pct": ticket_check.isna().mean() * 100,
    "unique_n": ticket_check.nunique(dropna=True)
})

variable_summary = variable_summary.sort_values(
    by="missing_pct",
    ascending=False
)

display(variable_summary)

#### Interpretation

The 100,000-row inspection shows that most variables are highly complete, but several fields require caution.

- `GroundDistance` is missing for about 65% of observations and is unlikely to be useful in the core model.
- `DwellTimeGroup` is missing for about 45% of observations and also has very high cardinality, so it should not be used without further investigation.
- `Break_LegacyLogic` has only one observed non-missing value and therefore provides no useful variation for regression.
- `Passengers` and `DollarCred` are constant in this sample and cannot contribute explanatory power.
- `RpYear`, `RpQuarter`, `RpMonth`, `SchFlYear`, and `OriginCountry` are also constant within this sample and are not useful as cross-sectional predictors.
- Several location identifiers contain hundreds of categories, while `AirportGroupString` contains more than 13,000 unique values. These variables would create very large dummy-variable sets and should not be included mechanically.
- The core fare, distance, itinerary, carrier, and purchase-window variables are almost completely observed and appear suitable for further consideration.

This inspection is based on the first 100,000 rows and is used only for structural assessment. The final modelling sample will later be drawn randomly from the full Ticket file.

### 2.4 Inspect Candidate Variable Values

In [ ]:
candidate_categorical = [
    "Coupons",
    "RoundTrip",
    "OnLine",
    "PurchaseWindowGroup",
    "RpCarrier",
    "IssuingCarrier",
    "ItinGeoType",
    "SchFlMonth"
]

for col in candidate_categorical:
    print(f"\n{col}")
    print("-" * len(col))
    print(
        ticket_check[col]
        .value_counts(dropna=False)
        .sort_index()
    )

In [ ]:
candidate_numeric = [
    "TotalAmount",
    "TaxAmount",
    "ItinYield",
    "TotalDistance",
    "MilesTraveled",
    "GroundDistance"
]

display(
    ticket_check[candidate_numeric]
    .describe()
    .T
)

#### Interpretation

The candidate variables show several useful patterns for the later regression analysis.

- `Coupons` is concentrated at one or two segments, although more complex itineraries are also present.
- `RoundTrip` is well balanced between one-way and round-trip tickets.
- `OnLine` is highly concentrated in the value 1, but the smaller value-0 group remains large enough to provide useful variation.
- `PurchaseWindowGroup` contains three clearly defined groups and appears suitable as a categorical predictor.
- `RpCarrier` contains 16 carriers, although several carriers have very small frequencies. Rare carrier categories may need to be consolidated later.
- `IssuingCarrier` contains slightly more categories than `RpCarrier` and appears closely related to it. Both should not be included automatically without assessing whether they provide distinct information.
- `ItinGeoType` is strongly concentrated in category 2 but still contains sufficient observations in category 1 for further consideration.
- `SchFlMonth` is almost entirely equal to 6, which is expected for a June reporting file. It provides essentially no useful cross-sectional variation and should not be considered a core predictor.

For the numerical variables:

- `TotalAmount` is strongly right-skewed, ranging from 0 to more than 14,000, so unusual fare values will require further investigation.
- `TotalDistance` and `MilesTraveled` have almost identical summary statistics and are likely to contain highly redundant information.
- `GroundDistance` is mostly zero when observed and is also heavily missing, making it unsuitable as a core predictor.
- `TaxAmount` and `ItinYield` are closely related to the ticket price itself and should not be treated automatically as explanatory variables for `TotalAmount`, as doing so may introduce mechanical relationships with the dependent variable.

### 2.5 Assess Redundancy Among Numerical Variables

Several numerical fields appear to measure closely related quantities. Before constructing the analysis dataset, their pairwise relationships are examined to avoid retaining multiple variables that contain essentially the same information.

In [ ]:
numeric_check = [
    "TotalAmount",
    "TaxAmount",
    "ItinYield",
    "TotalDistance",
    "MilesTraveled",
    "GroundDistance"
]

correlation_matrix = (
    ticket_check[numeric_check]
    .corr()
    .round(4)
)

display(correlation_matrix)

In [ ]:
distance_difference = (
    ticket_check["TotalDistance"]
    - ticket_check["MilesTraveled"]
)

print("TotalDistance - MilesTraveled")
print("--------------------------------")
print(distance_difference.describe())

print(
    f"\nExact matches: "
    f"{(distance_difference == 0).sum():,} "
    f"({(distance_difference == 0).mean() * 100:.2f}%)"
)

#### Interpretation

`TotalDistance` and `MilesTraveled` are almost perfectly correlated, with a correlation of 0.9997.

They are also exactly equal for 99.33% of observations in the 100,000-row inspection sample.

This indicates that the two variables contain essentially the same information for the purpose of this analysis. Retaining both would provide little additional explanatory value while introducing unnecessary multicollinearity.

`TotalDistance` will therefore be retained as the primary distance measure, while `MilesTraveled` will be excluded from the modelling dataset.

The high correlation between `TotalAmount` and `TaxAmount` also reinforces that `TaxAmount` should not be used as a conventional explanatory variable for ticket price because it is mechanically related to the amount paid.

## 3. Dependent Variable Assessment

Before constructing the modelling sample, the proposed response variable, `TotalAmount`, is examined in greater detail.

The objective is to identify unusual or special fare values and determine whether any observations require separate treatment before regression modelling.

### 3.1 Inspect the Distribution of TotalAmount

In [ ]:
fare_summary = ticket_check["TotalAmount"].describe(
    percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999]
)

display(fare_summary)

In [ ]:
print("Most common TotalAmount values:")
print(
    ticket_check["TotalAmount"]
    .value_counts(dropna=False)
    .head(25)
)

In [ ]:
fare_thresholds = pd.Series({
    "Missing": ticket_check["TotalAmount"].isna().sum(),
    "Equal to 0": (ticket_check["TotalAmount"] == 0).sum(),
    "Below 5.60": (ticket_check["TotalAmount"] < 5.60).sum(),
    "Equal to 5.60": (ticket_check["TotalAmount"] == 5.60).sum(),
    "Equal to 11.20": (ticket_check["TotalAmount"] == 11.20).sum(),
    "Below 20": (ticket_check["TotalAmount"] < 20).sum(),
    "Below 50": (ticket_check["TotalAmount"] < 50).sum(),
    "Above 1,000": (ticket_check["TotalAmount"] > 1000).sum(),
    "Above 2,000": (ticket_check["TotalAmount"] > 2000).sum()
})

display(fare_thresholds.to_frame("count"))

#### Interpretation

`TotalAmount` is strongly right-skewed. The median ticket amount is 216.00, compared with a mean of 301.09, while the upper tail extends beyond 14,000.

The lower tail also contains a noticeable concentration of unusually small amounts:

- 403 observations have `TotalAmount = 0`;
- 1,736 observations have `TotalAmount = 5.60`;
- 1,247 observations have `TotalAmount = 11.20`;
- 3,511 observations have `TotalAmount < 20`.

The values 5.60 and 11.20 occur far more frequently than neighbouring amounts, suggesting that they represent a distinct type of ticket record rather than ordinary variation in commercial fares.

At the upper end, 2,632 observations exceed 1,000 and 379 exceed 2,000. These should also be investigated rather than removed automatically.

No fare-based exclusion rule will be imposed at this stage. The unusually low and high observations will first be examined using the accompanying ticket variables.

### 3.2 Investigate Unusually Low Ticket Amounts

The unusually frequent low ticket amounts may represent a distinct type of record. Their accompanying fare components and itinerary characteristics are therefore examined before deciding whether they belong in the modelling population.

In [ ]:
ticket_check["AmountExclTax"] = (
    ticket_check["TotalAmount"]
    - ticket_check["TaxAmount"]
)

In [ ]:
low_fare_groups = (
    ticket_check[
        ticket_check["TotalAmount"].isin([0.00, 5.60, 11.20, 17.00])
    ]
    .groupby("TotalAmount")
    .agg(
        observations=("ItinID", "size"),
        mean_tax=("TaxAmount", "mean"),
        median_tax=("TaxAmount", "median"),
        mean_amount_excl_tax=("AmountExclTax", "mean"),
        median_amount_excl_tax=("AmountExclTax", "median"),
        mean_distance=("TotalDistance", "mean"),
        median_distance=("TotalDistance", "median"),
        mean_coupons=("Coupons", "mean")
    )
    .round(2)
)

display(low_fare_groups)

In [ ]:
low_fare_check = ticket_check[
    ticket_check["TotalAmount"] < 20
].copy()

print(f"Observations below 20: {len(low_fare_check):,}")

print(
    "TotalAmount exactly equals TaxAmount: "
    f"{(low_fare_check['TotalAmount'] == low_fare_check['TaxAmount']).sum():,} "
    f"({(low_fare_check['TotalAmount'] == low_fare_check['TaxAmount']).mean() * 100:.2f}%)"
)

print("\nMost common AmountExclTax values:")
print(
    low_fare_check["AmountExclTax"]
    .round(2)
    .value_counts()
    .head(15)
)

In [ ]:
print("Carrier distribution for TotalAmount < 20:")
print(
    low_fare_check["RpCarrier"]
    .value_counts()
    .head(15)
)

print("\nRoundTrip:")
print(low_fare_check["RoundTrip"].value_counts(dropna=False))

print("\nItinGeoType:")
print(low_fare_check["ItinGeoType"].value_counts(dropna=False))

print("\nPurchaseWindowGroup:")
print(low_fare_check["PurchaseWindowGroup"].value_counts(dropna=False))

#### Interpretation

The unusually low `TotalAmount` observations form a distinct group rather than the lower tail of ordinary commercial fares.

Among the 3,511 observations with `TotalAmount < 20`, 3,464 (98.66%) have:

$$
TotalAmount = TaxAmount,
$$

so their implied amount excluding tax is zero.

This pattern is especially clear for the common values 5.60, 11.20 and 17.00, where the median amount excluding tax is zero despite substantial travel distances.

These observations therefore appear to represent tax-only or otherwise special ticket records rather than conventionally priced tickets. They are distributed across several carriers, journey types and purchase-window groups, although they are particularly concentrated in `UA`.

For a project intended to model and benchmark ordinary airline fares, these records may belong to a different pricing population. However, an exclusion rule should not be based solely on an arbitrary fare threshold such as `TotalAmount < 20`.

Instead, the economically relevant characteristic is whether the ticket contains a positive non-tax fare component. This will be investigated across the broader sample before defining the final cleaning rule.

### 3.3 Examine Tax-Only Ticket Records

The previous analysis shows that most unusually low ticket amounts consist entirely of reported taxes. To determine whether this issue is limited to very low fares or occurs more broadly, the non-tax component of each ticket is examined across the full 100,000-row inspection sample.

In [ ]:
tax_only = (
    ticket_check["TotalAmount"].notna()
    & ticket_check["TaxAmount"].notna()
    & np.isclose(
        ticket_check["TotalAmount"],
        ticket_check["TaxAmount"]
    )
)

print(f"Tax-only observations: {tax_only.sum():,}")
print(f"Share of valid observations: {tax_only.mean() * 100:.2f}%")

In [ ]:
display(
    ticket_check.loc[tax_only, "TotalAmount"]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

In [ ]:
print("Most common TotalAmount values among tax-only records:")
print(
    ticket_check.loc[tax_only, "TotalAmount"]
    .value_counts()
    .head(20)
)

In [ ]:
tax_only_comparison = (
    ticket_check
    .assign(TaxOnly=np.where(tax_only, "Tax-only", "Positive fare component"))
    .groupby("TaxOnly")
    .agg(
        observations=("ItinID", "size"),
        mean_total_amount=("TotalAmount", "mean"),
        median_total_amount=("TotalAmount", "median"),
        mean_distance=("TotalDistance", "mean"),
        median_distance=("TotalDistance", "median"),
        mean_coupons=("Coupons", "mean")
    )
    .round(2)
)

display(tax_only_comparison)

#### Interpretation

Tax-only records account for 3,505 observations, or approximately 3.5% of the 100,000-row inspection sample.

These observations are clearly different from the rest of the ticket population:

- their median `TotalAmount` is only 5.60, compared with 223.00 for tickets with a positive non-tax fare component;
- the dominant amounts are 5.60, 11.20 and 0.00;
- despite these very small reported amounts, the median travel distance is 1,454 miles.

This confirms that the visible low-fare concentration is not simply ordinary fare variation. Instead, these records form a distinct group in which the reported ticket amount consists entirely of taxes.

For the intended fare-benchmarking analysis, the economically relevant population is likely to be tickets with a positive non-tax fare component. However, no observations will be removed based on this preliminary first-chunk inspection. A `TaxOnly` indicator will instead be retained during full-file sampling so that the pattern can be verified on the final random sample before any exclusion rule is applied.

## 4. Construct a Random Sample from the Full Ticket File

The previous sections used the first 100,000 observations only to inspect the structure and coding of the raw DB1C Ticket file.

Those observations are not a random sample because they simply correspond to the first records stored in the CSV file. They will therefore not be used as the final modelling dataset.

This section constructs a reproducible random sample from the entire Ticket file. A target of 100,000 observations is used to retain substantial cross-sectional variation while keeping subsequent exploratory analysis and regression modelling computationally manageable.

### 4.1 Define Variables to Retain

In [ ]:
ticket_cols = [
    # Identifier
    "ItinID",

    # Reporting / travel timing
    "RpYear",
    "RpMonth",
    "SchFlYear",
    "SchFlMonth",

    # Itinerary structure
    "Coupons",
    "RoundTrip",
    "OnLine",
    "ItinGeoType",

    # Origin / geography
    "Origin",
    "OriginCityMarketID",
    "OriginState",

    # Carrier
    "RpCarrier",
    "IssuingCarrier",

    # Purchase timing
    "PurchaseWindowGroup",

    # Fare information
    "TotalAmount",
    "TaxAmount",
    "ItinYield",

    # Distance
    "TotalDistance"
]

print(f"Variables retained: {len(ticket_cols)}")
print(ticket_cols)

### 4.2 Construct a Reproducible Random Sample from the Full File

Because the raw Ticket file is too large to load fully into memory, the file is processed sequentially in chunks.

To obtain a reproducible random sample of exactly 100,000 observations from the full dataset, each row is assigned a random priority using a fixed random seed. After all chunks have been processed, the 100,000 rows with the smallest random priorities are retained.

This gives every row in the source file an equal probability of being selected while avoiding the need to load the entire 2.52 GB dataset into memory.

In [ ]:
TARGET_SAMPLE_SIZE = 100_000
CHUNK_SIZE = 200_000
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)

sample_candidates = []
rows_processed = 0

for chunk in pd.read_csv(
    TICKET_FILE,
    usecols=ticket_cols,
    chunksize=CHUNK_SIZE,
    low_memory=False
):
    # Assign an independent random priority to each row
    chunk = chunk.copy()
    chunk["_random_priority"] = rng.random(len(chunk))

    sample_candidates.append(
        chunk.nsmallest(TARGET_SAMPLE_SIZE, "_random_priority")
    )

    rows_processed += len(chunk)

    # Periodically reduce memory usage by keeping only the current
    # best TARGET_SAMPLE_SIZE observations
    if len(sample_candidates) >= 5:
        combined = pd.concat(sample_candidates, ignore_index=True)

        sample_candidates = [
            combined.nsmallest(
                TARGET_SAMPLE_SIZE,
                "_random_priority"
            )
        ]

# Final reduction after the last chunk
ticket_sample = (
    pd.concat(sample_candidates, ignore_index=True)
    .nsmallest(TARGET_SAMPLE_SIZE, "_random_priority")
    .drop(columns="_random_priority")
    .reset_index(drop=True)
)

print(f"Rows processed from full file: {rows_processed:,}")
print(f"Final random sample size:      {len(ticket_sample):,}")
print(f"Number of columns retained:    {ticket_sample.shape[1]}")

#### Interpretation

The full June 2026 DB1C Ticket file contains 13,822,006 observations.

A reproducible random sample of exactly 100,000 ticket records has been drawn from the full file using a fixed random seed. Unlike the earlier inspection sample, this modelling sample is not restricted to the first rows of the CSV and is therefore more appropriate for subsequent exploratory analysis.

The sample retains 19 variables covering itinerary structure, timing, geography, carrier information, purchase timing, fare information, and travel distance.

### 4.3 Verify the Random Sample

In [ ]:
sample_summary = pd.DataFrame({
    "dtype": ticket_sample.dtypes.astype(str),
    "missing_n": ticket_sample.isna().sum(),
    "missing_pct": ticket_sample.isna().mean() * 100,
    "unique_n": ticket_sample.nunique(dropna=True)
}).sort_values(
    by="missing_pct",
    ascending=False
)

display(sample_summary)

In [ ]:
print("Carrier distribution:")
print(
    ticket_sample["RpCarrier"]
    .value_counts()
    .head(20)
)

print("\nPurchaseWindowGroup:")
print(
    ticket_sample["PurchaseWindowGroup"]
    .value_counts(dropna=False)
)

print("\nRoundTrip:")
print(
    ticket_sample["RoundTrip"]
    .value_counts(dropna=False)
)

print("\nOnLine:")
print(
    ticket_sample["OnLine"]
    .value_counts(dropna=False)
)

print("\nItinGeoType:")
print(
    ticket_sample["ItinGeoType"]
    .value_counts(dropna=False)
)

In [ ]:
display(
    ticket_sample[
        ["TotalAmount", "TaxAmount", "ItinYield", "TotalDistance", "Coupons"]
    ]
    .describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
    .T
)

#### Interpretation

The random sample differs substantially from the initial first-chunk inspection, confirming that the beginning of the raw CSV is not representative of the full Ticket file.

In particular, the carrier composition changes markedly. The random sample is dominated by major U.S. carriers such as `WN`, `AA`, `DL`, `UA`, `AS`, `F9`, and `B6`, whereas the first 100,000 rows were heavily concentrated in a small number of different carriers.

The random sample also contains:

- 49 reporting carriers instead of 16;
- 469 origin airports instead of 363;
- 438 origin city markets instead of 343;
- a higher mean travel distance of approximately 1,908 miles;
- a higher mean ticket amount of approximately 415.

The main variables remain highly complete. Only six observations are missing `TotalAmount`, `TaxAmount`, and `ItinYield`.

The fare distribution remains strongly right-skewed. The 1st percentile is 5.60 and the 5th percentile is 11.20, indicating that the unusually low fare records identified earlier remain present in the full-file random sample and should now be investigated using this representative sample.

This random sample will therefore replace the initial first-chunk data for all subsequent preparation and modelling work.

## 5. Data Quality Assessment and Cleaning

The random sample is now examined for missing values, special fare records, implausible observations, and variables that provide no useful cross-sectional variation.

Cleaning decisions will be based on identifiable data characteristics rather than arbitrary numerical cutoffs.

### 5.1 Identify Tax-Only Ticket Records

Preliminary inspection suggested that some unusually low ticket amounts consist entirely of reported taxes. This pattern is re-examined using the full-file random sample before deciding whether such observations should be excluded from the fare-modelling population.

In [ ]:
ticket_sample["AmountExclTax"] = (
    ticket_sample["TotalAmount"]
    - ticket_sample["TaxAmount"]
)

tax_only = (
    ticket_sample["TotalAmount"].notna()
    & ticket_sample["TaxAmount"].notna()
    & np.isclose(
        ticket_sample["TotalAmount"],
        ticket_sample["TaxAmount"]
    )
)

print(f"Tax-only observations: {tax_only.sum():,}")
print(
    f"Share of valid sample: "
    f"{tax_only.sum() / ticket_sample['TotalAmount'].notna().sum() * 100:.2f}%"
)

print("\nMost common TotalAmount values among tax-only records:")
print(
    ticket_sample.loc[tax_only, "TotalAmount"]
    .value_counts()
    .head(20)
)

In [ ]:
tax_only_comparison = (
    ticket_sample
    .assign(
        TaxOnly=np.where(
            tax_only,
            "Tax-only",
            "Positive fare component"
        )
    )
    .groupby("TaxOnly")
    .agg(
        observations=("ItinID", "size"),
        mean_total_amount=("TotalAmount", "mean"),
        median_total_amount=("TotalAmount", "median"),
        mean_distance=("TotalDistance", "mean"),
        median_distance=("TotalDistance", "median"),
        mean_coupons=("Coupons", "mean")
    )
    .round(2)
)

display(tax_only_comparison)

In [ ]:
print("Carrier distribution among tax-only records:")
print(
    ticket_sample.loc[tax_only, "RpCarrier"]
    .value_counts()
    .head(15)
)

print("\nPurchase-window distribution among tax-only records:")
print(
    ticket_sample.loc[tax_only, "PurchaseWindowGroup"]
    .value_counts(dropna=False)
)

#### Interpretation

The tax-only pattern remains clearly present in the full-file random sample.

A total of 9,339 observations, representing 9.34% of valid sampled tickets, have:

$$
TotalAmount = TaxAmount,
$$

and therefore have no positive non-tax fare component.

The two dominant reported amounts are 5.60 and 11.20, accounting for most tax-only records. These tickets nevertheless involve substantial travel distances, with a median distance of 1,474 miles.

Tax-only observations are also distributed across major carriers such as `WN`, `DL`, `AA`, and `UA`, and across all three purchase-window groups. The pattern is therefore not confined to a single carrier or market segment.

These records appear to represent a distinct fare population from conventionally priced tickets. Because the objective of the project is to explain and benchmark ordinary airline fares, tickets without a positive non-tax fare component are unlikely to be appropriate observations for the main regression sample.

Before applying an exclusion rule, the distribution of the non-tax fare component will be examined to determine whether other zero, negative, or extremely small values are present.

### 5.2 Examine the Non-Tax Fare Component

The non-tax component of the ticket amount is defined as:

$$
AmountExclTax = TotalAmount - TaxAmount.
$$

This measure is examined to determine whether observations with zero, negative, or unusually small fare components should be treated as part of the ordinary commercial fare population.

In [ ]:
display(
    ticket_sample["AmountExclTax"]
    .describe(
        percentiles=[
            0.01, 0.05, 0.10, 0.25, 0.50,
            0.75, 0.90, 0.95, 0.99
        ]
    )
)

In [ ]:
fare_component_counts = pd.Series({
    "Missing": ticket_sample["AmountExclTax"].isna().sum(),
    "Negative": (ticket_sample["AmountExclTax"] < 0).sum(),
    "Equal to 0": np.isclose(ticket_sample["AmountExclTax"], 0).sum(),
    "0 to 1": (
        (ticket_sample["AmountExclTax"] > 0)
        & (ticket_sample["AmountExclTax"] <= 1)
    ).sum(),
    "1 to 5": (
        (ticket_sample["AmountExclTax"] > 1)
        & (ticket_sample["AmountExclTax"] <= 5)
    ).sum(),
    "5 to 10": (
        (ticket_sample["AmountExclTax"] > 5)
        & (ticket_sample["AmountExclTax"] <= 10)
    ).sum(),
    "Above 10": (ticket_sample["AmountExclTax"] > 10).sum()
})

display(fare_component_counts.to_frame("count"))

In [ ]:
print("Most common AmountExclTax values:")
print(
    ticket_sample["AmountExclTax"]
    .round(2)
    .value_counts()
    .head(20)
)

#### Interpretation

The distribution of `AmountExclTax` confirms a distinct concentration at extremely small fare components.

- 9,339 observations have a zero non-tax fare component.
- A further 573 observations have a positive component no greater than 1.00.
- Of these, 500 observations have `AmountExclTax = 0.01`.
- Only a small number of observations fall between 1.00 and 10.00.
- The great majority of tickets, 89,944 observations, have a non-tax fare component above 10.00.

There are no negative values, so the issue is concentrated in zero or near-zero fare components rather than data inconsistencies caused by negative amounts.

The large frequency at exactly 0.01 suggests that some technically positive fare records may still represent special or nominally priced tickets. Therefore, defining valid commercial fares simply as `AmountExclTax > 0` would be too permissive.

The very small positive fare records should be examined before a final inclusion rule is established.

### 5.3 Investigate Near-Zero Fare Components

Tickets with a positive but extremely small non-tax fare component may represent nominally priced or special ticket records rather than ordinary commercial fares.

These observations are examined separately before determining the final fare-validity rule.

In [ ]:
near_zero_fare = ticket_sample[
    (ticket_sample["AmountExclTax"] > 0)
    & (ticket_sample["AmountExclTax"] <= 10)
].copy()

print(f"Near-zero positive fare observations: {len(near_zero_fare):,}")

print("\nMost common AmountExclTax values:")
print(
    near_zero_fare["AmountExclTax"]
    .round(2)
    .value_counts()
    .head(20)
)

In [ ]:
print("Most common TotalAmount / TaxAmount combinations:")
print(
    near_zero_fare
    .assign(
        AmountExclTax=near_zero_fare["AmountExclTax"].round(2)
    )
    [["TotalAmount", "TaxAmount", "AmountExclTax"]]
    .value_counts()
    .head(25)
)

In [ ]:
near_zero_summary = pd.DataFrame({
    "value": [
        len(near_zero_fare),
        near_zero_fare["TotalAmount"].mean(),
        near_zero_fare["TotalAmount"].median(),
        near_zero_fare["TotalDistance"].mean(),
        near_zero_fare["TotalDistance"].median(),
        near_zero_fare["Coupons"].mean()
    ]
},
    index=[
        "Observations",
        "Mean TotalAmount",
        "Median TotalAmount",
        "Mean TotalDistance",
        "Median TotalDistance",
        "Mean Coupons"
    ]
).round(2)

display(near_zero_summary)

In [ ]:
print("Carrier distribution:")
print(
    near_zero_fare["RpCarrier"]
    .value_counts()
    .head(15)
)

print("\nPurchaseWindowGroup:")
print(
    near_zero_fare["PurchaseWindowGroup"]
    .value_counts(dropna=False)
)

print("\nRoundTrip:")
print(
    near_zero_fare["RoundTrip"]
    .value_counts(dropna=False)
)

#### Interpretation

The near-zero positive fare records are not distributed smoothly between 0 and 10.

Instead, there is a strong concentration at nominal fare values:

- 500 observations have `AmountExclTax = 0.01`;
- 69 observations have `AmountExclTax = 0.02`;
- together, these account for most of the 711 observations with a positive fare component no greater than 10.

These nominal-fare records are also highly concentrated among `F9` and `G4`, whereas the remaining low positive fares are more dispersed.

This suggests that the 0.01 and 0.02 observations may represent a distinct pricing convention or special fare structure rather than ordinary continuous fare variation.

However, a general cutoff such as `AmountExclTax > 10` would be difficult to justify because some fares between 1 and 10 may represent genuine low-priced tickets.

The nominal-fare observations should therefore be separated from the broader low-fare group before defining the final inclusion rule.

### 5.4 Separate Nominal and Other Very Low Fare Components

The near-zero fare records are divided into nominal fare components of 0.01–0.02 and other low positive fare components above 0.02 but no greater than 10.

This distinction helps determine whether the unusual observations reflect a discrete pricing convention or simply the lower tail of ordinary fares.

In [ ]:
nominal_fare = ticket_sample[
    (ticket_sample["AmountExclTax"] > 0)
    & (ticket_sample["AmountExclTax"] <= 0.02)
].copy()

other_low_fare = ticket_sample[
    (ticket_sample["AmountExclTax"] > 0.02)
    & (ticket_sample["AmountExclTax"] <= 10)
].copy()

print(f"Nominal fare observations (0 < fare <= 0.02): {len(nominal_fare):,}")
print(f"Other low fare observations (0.02 < fare <= 10): {len(other_low_fare):,}")

In [ ]:
print("Nominal fare carrier distribution:")
print(
    nominal_fare["RpCarrier"]
    .value_counts()
    .head(15)
)

print("\nOther low fare carrier distribution:")
print(
    other_low_fare["RpCarrier"]
    .value_counts()
    .head(15)
)

In [ ]:
low_fare_comparison = pd.DataFrame({
    "Nominal fare": [
        len(nominal_fare),
        nominal_fare["AmountExclTax"].median(),
        nominal_fare["TotalAmount"].median(),
        nominal_fare["TotalDistance"].median(),
        nominal_fare["Coupons"].mean(),
        nominal_fare["RoundTrip"].mean()
    ],
    "Other low fare": [
        len(other_low_fare),
        other_low_fare["AmountExclTax"].median(),
        other_low_fare["TotalAmount"].median(),
        other_low_fare["TotalDistance"].median(),
        other_low_fare["Coupons"].mean(),
        other_low_fare["RoundTrip"].mean()
    ]
},
    index=[
        "Observations",
        "Median AmountExclTax",
        "Median TotalAmount",
        "Median TotalDistance",
        "Mean Coupons",
        "RoundTrip share"
    ]
).round(3)

display(low_fare_comparison)

#### Interpretation

The near-zero positive fare observations separate into two distinct groups.

There are 545 nominal-fare records with a non-tax fare component no greater than 0.02. These observations are overwhelmingly concentrated in `F9`, which accounts for 538 of the 545 records. Their median non-tax fare is only 0.01.

By contrast, only 166 observations have a non-tax fare component between 0.02 and 10. These observations are more dispersed across carriers and have a median non-tax fare of 5.63.

This indicates that the nominal 0.01–0.02 fares represent a discrete pricing pattern rather than a natural continuation of ordinary low fares.

A general minimum-fare cutoff such as 10 would therefore be inappropriate because it would also remove potentially genuine low-priced tickets. Instead, tax-only and nominal-fare records should be identified separately from the broader positive-fare population.

### 5.5 Correct Monetary Precision

Monetary variables are reported to the nearest cent, but floating-point arithmetic can introduce very small numerical precision errors when `TaxAmount` is subtracted from `TotalAmount`.

To ensure that fare classification reflects monetary values rather than floating-point representation, the derived non-tax fare component is rounded to two decimal places before applying the classification rules.

In [ ]:
ticket_sample["AmountExclTax"] = (
    ticket_sample["TotalAmount"]
    - ticket_sample["TaxAmount"]
).round(2)

### 5.6 Classify Fare Records

Based on the preceding data-quality assessment, ticket records are classified according to the size of their non-tax fare component.

Three groups are distinguished:

1. **Tax-only** — the ticket amount contains no positive non-tax fare component;
2. **Nominal fare** — the non-tax fare component is positive but no greater than 0.02;
3. **Regular positive fare** — the non-tax fare component exceeds 0.02.

The classification is retained as a data-quality flag so that special fare records can be identified explicitly before the main analysis sample is constructed.

In [ ]:
fare_conditions = [
    ticket_sample["AmountExclTax"].isna(),
    ticket_sample["AmountExclTax"] == 0,
    (ticket_sample["AmountExclTax"] > 0)
    & (ticket_sample["AmountExclTax"] <= 0.02),
    ticket_sample["AmountExclTax"] > 0.02
]

fare_labels = [
    "Missing",
    "Tax-only",
    "Nominal fare",
    "Regular positive fare"
]

ticket_sample["FareRecordType"] = np.select(
    fare_conditions,
    fare_labels,
    default="Review"
)

print(
    ticket_sample["FareRecordType"]
    .value_counts(dropna=False)
)

print("\nShare of sample (%):")
print(
    ticket_sample["FareRecordType"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

#### Interpretation

The fare-record classification shows that most observations belong to the ordinary positive-fare population:

- 90.09% are classified as `Regular positive fare`;
- 9.34% are `Tax-only`;
- 0.57% are `Nominal fare`;
- only 0.01% have missing fare information.

This confirms that the unusual fare records form a meaningful but clearly identifiable minority of the sample.

The `FareRecordType` variable will be retained as a data-quality flag. The main regression sample will focus on `Regular positive fare` observations, while tax-only, nominal-fare, and missing-fare records will be excluded explicitly rather than removed using an arbitrary fare cutoff.

### 5.7 Check for Duplicate Itineraries and Duplicate Records

Because `ItinID` is intended to identify individual ticket itineraries, duplicate identifiers or fully duplicated rows may indicate repeated records that should be investigated before constructing the final analysis dataset.

In [ ]:
duplicate_itinid = ticket_sample["ItinID"].duplicated(keep=False)
duplicate_rows = ticket_sample.duplicated(keep=False)

print(f"Duplicate ItinID observations: {duplicate_itinid.sum():,}")
print(f"Fully duplicated rows:         {duplicate_rows.sum():,}")

if duplicate_itinid.any():
    display(
        ticket_sample.loc[duplicate_itinid]
        .sort_values("ItinID")
        .head(30)
    )
else:
    print("\nNo duplicate ItinID values found.")

#### Interpretation

No duplicate `ItinID` values or fully duplicated rows are present in the 100,000-observation random sample.

This supports the interpretation that each sampled row represents a distinct itinerary-level ticket record. No duplicate-removal step is therefore required.

### 5.8 Construct the Main Analysis Sample

The main regression analysis is intended to explain ordinary commercial airfare variation.

Based on the preceding data-quality assessment, tax-only and nominal-fare records are treated as distinct pricing populations and excluded from the primary modelling sample. Observations with missing fare information are also excluded.

The resulting analysis sample therefore retains only tickets classified as `Regular positive fare`.

In [ ]:
analysis_sample = (
    ticket_sample[
        ticket_sample["FareRecordType"] == "Regular positive fare"
    ]
    .copy()
    .reset_index(drop=True)
)

print(f"Original random sample: {len(ticket_sample):,}")
print(f"Main analysis sample:   {len(analysis_sample):,}")
print(
    f"Share retained:         "
    f"{len(analysis_sample) / len(ticket_sample) * 100:.2f}%"
)

print("\nMinimum AmountExclTax in analysis sample:")
print(analysis_sample["AmountExclTax"].min())

In [ ]:
display(
    analysis_sample[
        ["TotalAmount", "TaxAmount", "AmountExclTax", "TotalDistance", "Coupons"]
    ]
    .describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
    .T
)

#### Interpretation

After excluding tax-only, nominal-fare, and missing-fare records, the main analysis sample contains 90,086 observations.

The minimum non-tax fare in the retained sample is 0.07, confirming that the nominal 0.01–0.02 fare records have been excluded correctly.

The resulting fare distribution is more consistent with the intended commercial-fare population:

- the extreme concentration at 5.60 and 11.20 is removed;
- substantial cross-sectional variation remains in ticket amount, distance, and itinerary complexity;
- the upper tail remains intact, including high-fare observations that may be genuine.

These high-fare observations are not removed during data preparation. Their influence will instead be assessed later using regression diagnostics such as studentized residuals, leverage, Cook's Distance, DFFITS, and DFBETAS.

The 90,086-observation sample will be used as the main dataset for subsequent exploratory analysis and regression modelling.

## 6. Final Variable Screening

Before exporting the cleaned dataset, the retained variables are reviewed to identify fields that are constant, nearly constant, redundant, or unsuitable for the main regression analysis.

This step removes only variables that are clearly uninformative or mechanically related to other variables. Formal regression-based variable selection will be conducted in later notebooks.

### 6.1 Review Constant and Near-Constant Variables

In [ ]:
final_variable_summary = pd.DataFrame({
    "dtype": analysis_sample.dtypes.astype(str),
    "missing_n": analysis_sample.isna().sum(),
    "missing_pct": analysis_sample.isna().mean() * 100,
    "unique_n": analysis_sample.nunique(dropna=True)
}).sort_values(
    by="unique_n"
)

display(final_variable_summary)

In [ ]:
for col in ["RpYear", "RpMonth", "SchFlYear", "SchFlMonth"]:
    print(f"\n{col}")
    print(analysis_sample[col].value_counts(dropna=False).sort_index())

#### Interpretation

Several variables provide no meaningful cross-sectional variation in the cleaned sample:

- `RpYear` is constant at 2026;
- `RpMonth` is constant at 6;
- `SchFlYear` is constant at 2026;
- `FareRecordType` is constant because the analysis sample contains only regular positive-fare observations.

`SchFlMonth` technically contains four values, but 90,100 of the original cleaned observations fall in June, with only a handful in May, July, or August. It therefore provides negligible useful variation and will not be used as a regression predictor.

These variables may still be retained as metadata where useful, but they will not form part of the modelling predictor set.

### 6.2 Compare Reporting and Issuing Carrier Variables

Both `RpCarrier` and `IssuingCarrier` identify airline-related characteristics, but including both may introduce redundant information.

Their agreement is examined before deciding which representation should be retained for modelling.

In [ ]:
carrier_match = (
    analysis_sample["RpCarrier"]
    == analysis_sample["IssuingCarrier"]
)

print(f"Exact carrier matches: {carrier_match.sum():,}")
print(
    f"Share matching: "
    f"{carrier_match.mean() * 100:.2f}%"
)

In [ ]:
carrier_cross = pd.crosstab(
    analysis_sample["RpCarrier"],
    analysis_sample["IssuingCarrier"]
)

display(carrier_cross)

In [ ]:
carrier_mismatch = analysis_sample.loc[
    ~carrier_match,
    ["RpCarrier", "IssuingCarrier"]
]

print("Most common mismatched carrier combinations:")
print(
    carrier_mismatch
    .value_counts()
    .head(20)
)

#### Interpretation

`RpCarrier` and `IssuingCarrier` contain almost identical information.

The two variables match exactly for 90,029 of the 90,086 observations, corresponding to 99.94% of the cleaned analysis sample. The carrier cross-tabulation is therefore overwhelmingly concentrated on the diagonal.

Only 57 observations contain different reporting and issuing carriers, and these mismatches are spread across a small number of carrier combinations.

Including both variables in the regression would therefore add almost no independent explanatory information while creating unnecessary redundancy.

`RpCarrier` will be retained as the primary carrier variable, while `IssuingCarrier` will be excluded from the modelling dataset.

### 6.3 Assess Geographic Variables

The dataset contains several alternative representations of origin geography:

- `Origin` identifies the origin airport;
- `OriginCityMarketID` identifies the broader origin city market;
- `OriginState` identifies the origin state.

These variables provide different levels of geographic detail. Their category frequencies are examined before deciding which geographic representation should be retained for subsequent modelling.

In [ ]:
geo_summary = pd.DataFrame({
    "unique_categories": [
        analysis_sample["Origin"].nunique(),
        analysis_sample["OriginCityMarketID"].nunique(),
        analysis_sample["OriginState"].nunique()
    ],
    "largest_category_n": [
        analysis_sample["Origin"].value_counts().iloc[0],
        analysis_sample["OriginCityMarketID"].value_counts().iloc[0],
        analysis_sample["OriginState"].value_counts().iloc[0]
    ],
    "smallest_category_n": [
        analysis_sample["Origin"].value_counts().iloc[-1],
        analysis_sample["OriginCityMarketID"].value_counts().iloc[-1],
        analysis_sample["OriginState"].value_counts().iloc[-1]
    ]
}, index=[
    "Origin",
    "OriginCityMarketID",
    "OriginState"
])

display(geo_summary)

In [ ]:
print("Top 15 origin airports:")
print(
    analysis_sample["Origin"]
    .value_counts()
    .head(15)
)

print("\nTop 15 origin city markets:")
print(
    analysis_sample["OriginCityMarketID"]
    .value_counts()
    .head(15)
)

print("\nOrigin state distribution:")
print(
    analysis_sample["OriginState"]
    .value_counts()
)

#### Interpretation

The three origin variables provide different levels of geographic detail.

- `Origin` contains 467 airport categories.
- `OriginCityMarketID` contains 436 city-market categories.
- `OriginState` contains 53 state or territory categories.

Using airport- or city-market-level indicators in the initial regression would require several hundred dummy variables, making the model unnecessarily complex and difficult to interpret.

`OriginState` provides a more practical geographic representation. It retains meaningful cross-sectional variation while keeping the number of categorical levels manageable.

Accordingly:

- `OriginState` will be retained as the primary geographic candidate for the initial regression model;
- `Origin` and `OriginCityMarketID` will remain in the prepared dataset for descriptive analysis and possible later sensitivity checks, but will not be included automatically in the initial broad model.

### 6.4 Assess Carrier Category Sparsity

Although `RpCarrier` is retained as the primary carrier variable, some carrier categories may be represented by very few observations.

Rare categories can produce unstable dummy-variable estimates and make interpretation unnecessarily difficult. Their frequencies are therefore examined before deciding whether carrier categories should be consolidated later.

In [ ]:
carrier_counts = (
    analysis_sample["RpCarrier"]
    .value_counts()
    .sort_values(ascending=False)
)

display(carrier_counts.to_frame("count"))

In [ ]:
carrier_frequency_summary = pd.Series({
    "Number of carriers": len(carrier_counts),
    "Carriers with < 10 observations": (carrier_counts < 10).sum(),
    "Carriers with < 25 observations": (carrier_counts < 25).sum(),
    "Carriers with < 50 observations": (carrier_counts < 50).sum(),
    "Carriers with < 100 observations": (carrier_counts < 100).sum(),
    "Observations in carriers with < 50": carrier_counts[carrier_counts < 50].sum(),
    "Observations in carriers with < 100": carrier_counts[carrier_counts < 100].sum()
})

display(carrier_frequency_summary.to_frame("value"))

#### Interpretation

`RpCarrier` contains 44 carrier categories, but the distribution is highly concentrated.

The major carriers account for nearly all observations, while 33 carriers contain fewer than 100 observations each. Collectively, these rare categories represent only 388 observations, or approximately 0.43% of the cleaned sample.

Very small carrier groups may produce unstable dummy-variable estimates in the regression model. However, grouping rare carriers requires a modelling threshold that is partly discretionary.

The original `RpCarrier` variable will therefore be preserved in the prepared dataset. Any consolidation of rare carriers into an `Other` category will be considered later when the initial broad regression model is constructed.

### 6.5 Verify Categorical Variable Coding

The remaining low-cardinality categorical predictors are checked once more in the cleaned analysis sample to confirm that their coding is complete and suitable for subsequent regression modelling.

In [ ]:
categorical_check = [
    "RoundTrip",
    "OnLine",
    "ItinGeoType",
    "PurchaseWindowGroup"
]

for col in categorical_check:
    print(f"\n{col}")
    print("-" * len(col))

    counts = analysis_sample[col].value_counts(
        dropna=False
    ).sort_index()

    percentages = analysis_sample[col].value_counts(
        normalize=True,
        dropna=False
    ).sort_index().mul(100).round(2)

    display(
        pd.DataFrame({
            "count": counts,
            "percent": percentages
        })
    )

In [ ]:
coupon_distribution = pd.DataFrame({
    "count": analysis_sample["Coupons"].value_counts().sort_index(),
    "percent": (
        analysis_sample["Coupons"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )
})

display(coupon_distribution)

#### Interpretation

The remaining low-cardinality predictors contain sufficient variation for subsequent regression modelling.

- `RoundTrip` is relatively balanced, with 54.96% round-trip and 45.04% one-way itineraries.
- `OnLine` is more concentrated, with 86.93% of observations coded as 1, but the 13.07% coded as 0 still provides substantial variation.
- `ItinGeoType` is strongly concentrated in category 2 (92.13%), although category 1 still contains 7,090 observations and can therefore be retained for further consideration.
- `PurchaseWindowGroup` contains three meaningful groups, with sufficient observations in each category.
- `Coupons` is concentrated at one or two coupons, but more complex itineraries are also represented. It will be retained as a numerical count variable initially, with its functional form assessed later if necessary.

No additional cleaning is required for these variables at the data-preparation stage.

### 6.6 Define the Initial Candidate Predictor Set

Based on the preceding screening, eight variables are designated as the initial candidate predictors for regression modelling:

- `TotalDistance`;
- `Coupons`;
- `RoundTrip`;
- `OnLine`;
- `PurchaseWindowGroup`;
- `RpCarrier`;
- `ItinGeoType`;
- `OriginState`.

These variables represent distinct and interpretable dimensions of itinerary structure, purchase characteristics, carrier identity, and geography, while retaining sufficient cross-sectional variation for modelling.

The candidate set is defined through data-quality, redundancy, dimensionality, and interpretability considerations rather than formal regression-based selection. In particular, `Origin` and `OriginCityMarketID` are retained only as supplementary geographic variables because their several hundred categories would substantially increase model complexity relative to the state-level representation.

Formal assessment of the contribution of these candidate predictors is deferred to the regression modelling stage.

## 7. Construct Final Prepared Dataset

The cleaned analysis sample is now reduced to a set of variables that will be carried forward to subsequent notebooks.

Variables are retained for one of three purposes:

1. **identification and traceability**;
2. **exploratory analysis or regression modelling**;
3. **fare-quality and descriptive checks**.

Variables that are constant, mechanically redundant, severely duplicated by another field, or unsuitable for the intended regression model are excluded.

### 7.1 Define Final Variables

In [ ]:
final_columns = [
    # Identifier
    "ItinID",

    # Dependent variable
    "TotalAmount",

    # Main candidate predictors
    "TotalDistance",
    "Coupons",
    "RoundTrip",
    "OnLine",
    "PurchaseWindowGroup",
    "RpCarrier",
    "ItinGeoType",
    "OriginState",

    # Additional geography for EDA / sensitivity analysis
    "Origin",
    "OriginCityMarketID",

    # Fare-related variables retained for descriptive checks only
    "TaxAmount",
    "AmountExclTax",

    # Metadata
    "RpYear",
    "RpMonth"
]

prepared_data = (
    analysis_sample[final_columns]
    .copy()
    .reset_index(drop=True)
)

print(f"Final prepared rows:    {prepared_data.shape[0]:,}")
print(f"Final prepared columns: {prepared_data.shape[1]}")

display(prepared_data.head())

#### Interpretation

The prepared dataset contains 90,086 cleaned ticket observations and 16 retained variables.

The dataset preserves the original itinerary identifier, the dependent variable `TotalAmount`, the main candidate predictors, supplementary geographic information for exploratory analysis, and fare-component variables used to document the cleaning process.

Before saving the dataset, the remaining variable types and data integrity will be checked one final time.

### 7.2 Standardize Variable Types

In [ ]:
prepared_data["RoundTrip"] = prepared_data["RoundTrip"].astype("int64")
prepared_data["OnLine"] = prepared_data["OnLine"].astype("int64")
prepared_data["OriginCityMarketID"] = prepared_data["OriginCityMarketID"].astype("int64")

### 7.3 Final Integrity Check

In [ ]:
print(f"Rows:    {prepared_data.shape[0]:,}")
print(f"Columns: {prepared_data.shape[1]}")

print("\nMissing values:")
print(prepared_data.isna().sum())

print(f"\nDuplicate ItinID values: {prepared_data['ItinID'].duplicated().sum():,}")
print(f"Fully duplicated rows:   {prepared_data.duplicated().sum():,}")

print("\nData types:")
print(prepared_data.dtypes)

In [ ]:
final_range_check = pd.DataFrame({
    "min": prepared_data[
        ["TotalAmount", "TotalDistance", "Coupons", "AmountExclTax"]
    ].min(),
    "max": prepared_data[
        ["TotalAmount", "TotalDistance", "Coupons", "AmountExclTax"]
    ].max()
})

display(final_range_check)

#### Interpretation

The final prepared dataset passes the integrity checks.

- All 90,086 observations are complete across the 16 retained variables.
- No duplicate `ItinID` values are present.
- No fully duplicated rows are present.
- Binary indicators such as `RoundTrip` and `OnLine` are stored as integers.
- `OriginCityMarketID` is stored as an integer identifier.
- The retained numerical variables have plausible ranges, with no negative fare or distance values.

The dataset is therefore ready to be exported for use in subsequent exploratory and regression notebooks.

## 8. Result Export / Saving

The cleaned and screened analysis dataset is saved to the processed-data directory so that subsequent notebooks can load the same prepared sample without repeating the raw-file processing and cleaning steps.

### 8.1 Save Prepared Dataset

In [ ]:
csv_output = PROCESSED_DATA_DIR / "db1c_ticket_prepared_202606.csv"
parquet_output = PROCESSED_DATA_DIR / "db1c_ticket_prepared_202606.parquet"

prepared_data.to_csv(
    csv_output,
    index=False
)

prepared_data.to_parquet(
    parquet_output,
    index=False
)

print(f"CSV saved to:     {csv_output}")
print(f"Parquet saved to: {parquet_output}")

In [ ]:
print(f"CSV exists:     {csv_output.exists()}")
print(f"Parquet exists: {parquet_output.exists()}")

if csv_output.exists():
    print(
        f"CSV size:       "
        f"{csv_output.stat().st_size / (1024 ** 2):.2f} MB"
    )

if parquet_output.exists():
    print(
        f"Parquet size:   "
        f"{parquet_output.stat().st_size / (1024 ** 2):.2f} MB"
    )

#### Interpretation

The prepared dataset was successfully exported in both CSV and Parquet formats.

- CSV size: 6.67 MB
- Parquet size: 1.97 MB

The Parquet version is substantially more storage-efficient and preserves data types more reliably, so it will be used as the default input for subsequent notebooks.

The CSV version is retained for portability and manual inspection.

## 9. Conclusion

Notebook 01 prepared a clean, reproducible cross-sectional sample from the June 2026 DB1C Ticket dataset.

The raw 2.52 GB Ticket file contained 13,822,006 observations. A reproducible random sample of 100,000 tickets was drawn from the full file to avoid dependence on the ordering of the raw CSV.

The data-preparation process identified several important issues:

- tax-only ticket records formed a distinct fare population;
- nominal non-tax fares of 0.01–0.02 represented a separate pricing pattern;
- `TotalDistance` and `MilesTraveled` were effectively redundant;
- `RpCarrier` and `IssuingCarrier` contained almost identical information;
- several timing variables were constant or nearly constant;
- high-cardinality geographic variables were retained only for descriptive or sensitivity analysis.

The main analysis sample excludes tax-only, nominal-fare, and missing-fare observations and contains:

- **90,086 observations**
- **16 retained variables**
- **no missing values**
- **no duplicate itinerary IDs**
- **no fully duplicated rows**

The prepared dataset is saved in both CSV and Parquet formats and will be used as the common input for subsequent exploratory analysis and regression modelling.

## 10. processed data from parquet

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns